# DiffusionRig grid generation

In [ ]:
import numpy as np
import os, glob
import matplotlib.pyplot as plt
from PIL import Image
import json

mani_light = 'rotate_sh_axis=1'
sample_pair_json = '/home/mint/Dev/DiFaReli++/difareli_pp/experiment_scripts/TPAMI/sample_json/TPAMI_MajorRevision/rotateSH.json'
num_frames = 60
with open(sample_pair_json, 'r') as f:
    sample_pairs = json.load(f)['pair']
    sample_pairs_k = [k for k in sample_pairs.keys()]
    sample_pairs_v = [v for v in sample_pairs.values()]


def gen(fid, src, dst, fn):
    rows = []
    for scale_sh in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
        cols = []
        for fi in fid:
            sampling_path = f'/data/mint/TPAMI_MajorRevision/New_Baselines_Tuning/DiffusionRig/ffhq_tuning/{mani_light}/src={src}_dst={dst}/scale_sh={scale_sh}/n_step={num_frames}/'
            imgs = sorted(glob.glob(f'{sampling_path}/{fn}_frame*.png'))
            if len(imgs) == 0:
                frame = None
            else:
                frame = imgs[fi]
            cols.append(frame)
        rows.append(cols)
    return rows

for idx in [0, 1]:
    pair = sample_pairs_v[idx]
    pair_id = sample_pairs_k[idx]
    src = pair['src']
    dst = pair['dst']
    os.makedirs(f'./tuning_grid_diffusionrig/{mani_light}_grid_{src}/', exist_ok=True)
    
    frame_id = [[3, 20, 33, 50, 55], [2, 13, 38, 48, 54]]
    for fn in ['res', 'ren']:
        for fi in frame_id:
            fn_grid = gen(fi, src, dst, fn=fn)
            
            # Load image
            out_grid = []
            for i in range(len(fn_grid)):
                col = []
                for j in range(len(fn_grid[i])):
                    if fn_grid[i][j] is not None:
                        img = Image.open(fn_grid[i][j])
                        img = img.resize((256, 256), Image.LANCZOS)
                    else:
                        img = Image.new('RGB', (256, 256), color='black')
                    col.append(np.array(img))
                out_grid.append(np.concatenate(col, axis=1))
            out_grid = np.concatenate(out_grid, axis=0)
            
            # plt.figure(figsize=(10, 10))
            # plt.imshow(out_grid)
            # plt.show()
            # plt.close()
            
            Image.fromarray(out_grid).save(f'./tuning_grid_diffusionrig/{mani_light}_grid_{src}/{fn}_{src}.png')

# Videos

In [2]:
import numpy as np
import os, glob
import matplotlib.pyplot as plt
from PIL import Image
import json
import subprocess

mani_light = 'rotate_sh_axis=1'
sample_pair_json = '/home/mint/Dev/DiFaReli++/difareli_pp/experiment_scripts/TPAMI/sample_json/TPAMI_MajorRevision/rotateSH.json'
num_frames = 60
with open(sample_pair_json, 'r') as f:
    sample_pairs = json.load(f)['pair']
    sample_pairs_k = [k for k in sample_pairs.keys()]
    sample_pairs_v = [v for v in sample_pairs.values()]


def gen(src, dst, fn):
    rows = []
    for scale_sh in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
        sampling_path = f'/data/mint/TPAMI_MajorRevision/New_Baselines_Tuning/DiffusionRig/ffhq_tuning/{mani_light}/src={src}_dst={dst}/scale_sh={scale_sh}/n_step={num_frames}/'
        vid = f'{sampling_path}/{fn}_rt.mp4'
        rows.append(vid)
    return rows

import shutil

def hstack_videos(paths, out_path, fps=None, size=256, quiet=True):
    assert paths and len(paths) >= 1, "paths must be a non-empty list"
    for p in paths:
        if not os.path.exists(p):
            raise FileNotFoundError(p)

    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg not found on PATH")

    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)

    # Inputs
    input_vids = " ".join(f'-i "{p}"' for p in paths)

    # Filters: scale each, format unify
    filters = []
    for i in range(len(paths)):
        filters.append(f"[{i}:v]scale=-2:{size},format=yuv420p[v{i}]")

    # Stack
    filters.append("".join(f"[v{i}]" for i in range(len(paths))) + f"hstack=inputs={len(paths)}[stacked]")

    # Optional fps: chain after stacked
    if fps:
        filters.append(f"[stacked]fps=fps={fps}[out]")
        map_label = "[out]"
    else:
        filters.append("[stacked]copy[out]")
        map_label = "[out]"

    filter_complex = ";".join(filters)

    quiet_opt = "-hide_banner -loglevel error" if quiet else ""

    cmd = (
        f'ffmpeg -y {input_vids} '
        f"-filter_complex '{filter_complex}' "
        f"-map '{map_label}' -c:v libx264 -pix_fmt yuv420p -shortest "
        f"{quiet_opt} \"{out_path}\" 2>ffmpeg_stderr.txt"
    )

    print("Running:", cmd)
    ret = os.system(cmd)
    if ret != 0:
        raise RuntimeError("ffmpeg failed. See ffmpeg_stderr.txt")


for idx in [0, 1]:
    pair = sample_pairs_v[idx]
    pair_id = sample_pairs_k[idx]
    src = pair['src']
    dst = pair['dst']
    os.makedirs(f'./tuning_grid_diffusionrig/{mani_light}_grid_{src}/', exist_ok=True)
    
    for fn in ['res', 'ren']:
        fn_grid = gen(src, dst, fn=fn)
        hstack_videos(fn_grid, f'./tuning_grid_diffusionrig/{mani_light}_grid_{src}/{fn}_{src}.mp4', fps=24)

Running: ffmpeg -y -i "/data/mint/TPAMI_MajorRevision/New_Baselines_Tuning/DiffusionRig/ffhq_tuning/rotate_sh_axis=1/src=60684.jpg_dst=60000.jpg/scale_sh=0.5/n_step=60//res_rt.mp4" -i "/data/mint/TPAMI_MajorRevision/New_Baselines_Tuning/DiffusionRig/ffhq_tuning/rotate_sh_axis=1/src=60684.jpg_dst=60000.jpg/scale_sh=0.6/n_step=60//res_rt.mp4" -i "/data/mint/TPAMI_MajorRevision/New_Baselines_Tuning/DiffusionRig/ffhq_tuning/rotate_sh_axis=1/src=60684.jpg_dst=60000.jpg/scale_sh=0.7/n_step=60//res_rt.mp4" -i "/data/mint/TPAMI_MajorRevision/New_Baselines_Tuning/DiffusionRig/ffhq_tuning/rotate_sh_axis=1/src=60684.jpg_dst=60000.jpg/scale_sh=0.8/n_step=60//res_rt.mp4" -i "/data/mint/TPAMI_MajorRevision/New_Baselines_Tuning/DiffusionRig/ffhq_tuning/rotate_sh_axis=1/src=60684.jpg_dst=60000.jpg/scale_sh=0.9/n_step=60//res_rt.mp4" -i "/data/mint/TPAMI_MajorRevision/New_Baselines_Tuning/DiffusionRig/ffhq_tuning/rotate_sh_axis=1/src=60684.jpg_dst=60000.jpg/scale_sh=1.0/n_step=60//res_rt.mp4" -filter_c